# CS604 dataset inspection
Run in Colab from top to bottom. CPU is sufficient: this notebook does not train a model or load a checkpoint. It reads images and saves tables and galleries to Drive. Official validation is unlabeled; internal holdout labels come only from training data.

Start with the sampled scan. Set `SCAN_PER_SPLIT = None` for a complete scan (slower). Duplicate conclusions apply only to scanned images. No images or labels are modified.


In [ ]:
# Settings
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
BRANCH = "dataset-inspection"  # Must contain this notebook and the shared trainer.
REPO_DIR = "/content/cs604-project"
DATA_DIR = "/content/data"  # Folder directly containing train.txt and val.txt.
DOWNLOAD_DATA = True        # False if images already exist at DATA_DIR.
MOUNT_DRIVE = True
OUTPUT_DIR = "/content/drive/MyDrive/CS604_RUNS/dataset_inspection"
CONFIG = "configs/bitfit_cosine_448.yaml"
SEED = 42
SCAN_PER_SPLIT = 5000       # None scans every manifest image.
HOLDOUT_PER_CLASS = 5
CLASS_IDS = []              # Empty chooses six classes reproducibly.
CROP_PATHS = []             # Optional manifest-relative paths; empty selects examples.
HASH_EXACT = True          # SHA256 of file bytes; adds disk I/O.
RUN_NEAR_DUPLICATES = True # dHash candidates, not proof of duplicates.
NEAR_DISTANCE = 4
MAX_NEAR_PAIRS = 200


In [ ]:
from pathlib import Path
import os, subprocess, sys
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
repo = Path(REPO_DIR)
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
if DOWNLOAD_DATA and not (Path(DATA_DIR) / 'train.txt').exists():
    subprocess.run([sys.executable, 'scripts/download_data.py', '--dest', DATA_DIR], check=True)
assert (Path(DATA_DIR) / 'train.txt').is_file(), 'Set DATA_DIR to the folder containing train.txt.'
out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)
print('Outputs:', out)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## 1. Manifest integrity and a reproducible internal holdout
Check class counts, invalid labels, repeated paths and paths shared across splits. The holdout uses the same seeded pandas shuffle as the cached linear probe. Exported files list membership only; they do not create a new training dataset. For future experiments, fit prototypes and train exclusively on `internal_fit.csv` membership, never on holdout images.


In [ ]:
import hashlib, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageFilter
from tqdm.auto import tqdm
random.seed(SEED)
np.random.seed(SEED)
root = Path(DATA_DIR)

def read_manifest(split):
    rows = []
    manifest = root / f'{split}.txt'
    if not manifest.exists():
        return pd.DataFrame(columns=['split', 'path', 'label'])
    for line_number, line in enumerate(manifest.read_text().splitlines(), 1):
        parts = line.split()
        if not parts:
            continue
        if len(parts) not in (1, 2):
            raise ValueError(f'{manifest}:{line_number}: expected path [label]')
        rows.append((split, parts[0], int(parts[1]) if len(parts) == 2 else -1))
    return pd.DataFrame(rows, columns=['split', 'path', 'label'])

manifest = pd.concat([read_manifest(s) for s in ['train', 'val', 'test']], ignore_index=True)
train = manifest[manifest.split.eq('train')].copy()
assert len(train) and train.label.ge(0).all(), 'Training images must have labels.'
summary = manifest.groupby('split').agg(images=('path','size'), labeled=('label', lambda x: int(x.ge(0).sum())), unique_paths=('path','nunique'))
display(summary)
counts = train.groupby('label').size().rename('images')
display(counts.describe())
print('Expected course dataset: 450,000 training images, 10,000 classes, 45 per class.')
print('Observed:', len(train), 'images;', len(counts), 'classes')
print('Labels outside [0, 9999]:', int((~train.label.between(0,9999)).sum()))
print('Missing class IDs:', len(set(range(10000)) - set(counts.index)))
print('Classes with counts other than 45:', int(counts.ne(45).sum()))
path_duplicates = manifest[manifest.duplicated('path', keep=False)].sort_values('path')
path_duplicates.to_csv(out / 'repeated_manifest_paths.csv', index=False)
display(path_duplicates.head(20))
counts.to_csv(out / 'class_counts.csv')
shuffled = train.sample(frac=1, random_state=SEED)
holdout_indices = shuffled.index[shuffled.groupby('label').cumcount() < HOLDOUT_PER_CLASS]
train['partition'] = np.where(train.index.isin(holdout_indices), 'holdout', 'fit')
assert train.groupby('label').partition.apply(lambda s: (s == 'fit').sum()).gt(0).all(), 'Holdout leaves a class without fit images.'
for partition in ['fit', 'holdout']:
    train[train.partition.eq(partition)].to_csv(out / f'internal_{partition}.csv', index=False)
print(train.partition.value_counts())
settings = {k: globals()[k] for k in ['DATA_DIR','CONFIG','SEED','SCAN_PER_SPLIT','HOLDOUT_PER_CLASS','HASH_EXACT','NEAR_DISTANCE','MAX_NEAR_PAIRS']}
settings['commit'] = subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip()
settings['manifest_sha256'] = {s: hashlib.sha256((root / f'{s}.txt').read_bytes()).hexdigest() for s in manifest.split.unique()}
(out / 'settings.json').write_text(json.dumps(settings, indent=2))


## 2. Image scan and train/validation comparison
The sample is uniform within each split. This scan decodes images, records missing/corrupt files, measures dimensions and simple appearance proxies. Edge variation is a rough sharpness proxy, influenced by resolution and content; it is not a label for bad images. dHash uses displayed orientation; statistics and crop galleries use raw orientation to match the trainer's PIL loading behavior.


In [ ]:
def dhash(image):
    a = np.asarray(ImageOps.exif_transpose(image).convert('L').resize((9,8)), dtype=np.int16)
    return sum(int(bit) << i for i, bit in enumerate((a[:,1:] > a[:,:-1]).ravel()))

selected = pd.concat([g if SCAN_PER_SPLIT is None else g.sample(n=min(len(g), SCAN_PER_SPLIT), random_state=SEED) for _, g in manifest.groupby('split')], ignore_index=True)
records = []
for row in tqdm(selected.itertuples(index=False), total=len(selected), desc='Scanning images'):
    record = {'split': row.split, 'path': row.path, 'label': row.label}
    try:
        path = root / row.path
        with Image.open(path) as image:
            image.load()
            rgb = image.convert('RGB')
            gray = rgb.convert('L').resize((128,128))
            pixels = np.asarray(gray, dtype=float)
            record.update(width=image.width, height=image.height, aspect=image.width/image.height,
                          mode=image.mode, exif_orientation=image.getexif().get(274,1),
                          brightness=pixels.mean(), contrast=pixels.std(),
                          edge_variance=np.asarray(gray.filter(ImageFilter.FIND_EDGES), dtype=float)[1:-1,1:-1].var(),
                          dhash=f'{dhash(image):016x}', bytes=path.stat().st_size)
        if HASH_EXACT:
            h = hashlib.sha256()
            with path.open('rb') as handle:
                for chunk in iter(lambda: handle.read(1024*1024), b''):
                    h.update(chunk)
            record['sha256'] = h.hexdigest()
        record['error'] = ''
    except Exception as exc:
        record['error'] = f'{type(exc).__name__}: {exc}'
    records.append(record)
stats = pd.DataFrame(records)
stats.to_csv(out / 'image_stats.csv', index=False)
errors = stats[stats.error.ne('')]
errors.to_csv(out / 'image_errors.csv', index=False)
good = stats[stats.error.eq('')].copy()
assert len(good), 'No images decoded. Check DATA_DIR and image paths.'
print('Scanned:', len(stats), 'Failures:', len(errors))
display(errors.head(20))
display(good.groupby('split')[['width','height','aspect','brightness','contrast','edge_variance']].median())
fig, axes = plt.subplots(2,3, figsize=(15,8))
for ax, field in zip(axes.flat, ['width','height','aspect','brightness','contrast','edge_variance']):
    for split, group in good.groupby('split'):
        ax.hist(group[field], bins=40, density=True, alpha=.4, label=split)
    ax.set_title(field); ax.legend()
fig.tight_layout(); fig.savefig(out / 'split_distributions.png', dpi=150); plt.show()
print('Non-default EXIF orientation:', int(good.exif_orientation.ne(1).sum()))


## 3. Class galleries and unusual examples
Look for tiny objects, multiple objects, text, inconsistent labels, and classes that differ only in fine details. Class IDs are numeric; this repo does not supply semantic class names. Low edge variation can be a naturally smooth image, so review visually before deciding anything.


In [ ]:
def gallery(rows, filename, columns=5):
    rows = list(rows)
    if not rows:
        print('No examples:', filename); return
    fig, axes = plt.subplots((len(rows)+columns-1)//columns, columns, figsize=(columns*3, ((len(rows)+columns-1)//columns)*3), squeeze=False)
    for ax in axes.flat: ax.axis('off')
    for ax, row in zip(axes.flat, rows):
        try:
            with Image.open(root / row['path']) as im: ax.imshow(im.convert('RGB'))
            ax.set_title(f"{row['split']} | class {row['label']}\n{Path(row['path']).name}", fontsize=8)
        except Exception as exc: ax.set_title(str(exc)[:80], fontsize=8)
    fig.tight_layout(); fig.savefig(out / filename, dpi=130); plt.show()

classes = CLASS_IDS or sorted(random.Random(SEED).sample(sorted(train.label.unique().tolist()), min(6, train.label.nunique())))
for class_id in classes:
    group = train[train.label.eq(class_id)]
    gallery(group.sample(min(15,len(group)), random_state=SEED).to_dict('records'), f'class_{class_id}.png')
for split, group in good.groupby('split'):
    gallery(group.sample(min(10,len(group)), random_state=SEED).to_dict('records'), f'{split}_random.png')
    gallery(group.nsmallest(10,'edge_variance').to_dict('records'), f'{split}_low_edge_variation.png')
    gallery(group.assign(short_side=group[['width','height']].min(axis=1)).nsmallest(10,'short_side').to_dict('records'), f'{split}_small_images.png')


## 4. Actual training crops versus evaluation crops
These transforms come directly from `src.data.build_transforms` for your selected config. Every row shows an original image, three independently drawn training crops, and evaluation crops at 0.90/0.75/0.60. Normalization is reversed for display. A smaller evaluation crop fraction includes more surrounding context after resizing; inspect what happens to small objects. Flip TTA mirrors the resulting crop and does not recover content outside it.


In [ ]:
import torch
from timm.data import resolve_data_config
from timm.models import get_pretrained_cfg
from src.utils import load_config, set_seed
from src.data import build_transforms
cfg = load_config(CONFIG)
train_tf, eval_tf = build_transforms(cfg)
print('Training transform:', train_tf)
print('Evaluation transform:', eval_tf)
pc = get_pretrained_cfg(cfg.model_name)
dc = resolve_data_config({'input_size': (3,cfg.img_size,cfg.img_size)}, pretrained_cfg=pc.to_dict() if hasattr(pc,'to_dict') else pc)
mean = torch.tensor(dc['mean'])[:,None,None]
std = torch.tensor(dc['std'])[:,None,None]
def view_tensor(t): return (t.cpu()*std+mean).clamp(0,1).permute(1,2,0).numpy()
paths = CROP_PATHS or good.sample(min(6,len(good)), random_state=SEED).path.tolist()
set_seed(SEED)
fig, axes = plt.subplots(len(paths),7, figsize=(21,3*len(paths)), squeeze=False)
for axs, path in zip(axes, paths):
    with Image.open(root/path) as im: image = im.convert('RGB')
    axs[0].imshow(image); axs[0].set_title('Original\n'+Path(path).name, fontsize=8)
    for i in range(3):
        axs[i+1].imshow(view_tensor(train_tf(image))); axs[i+1].set_title(f'Train crop {i+1}')
    for ax, crop in zip(axs[4:], [.90,.75,.60]):
        _, tf = build_transforms(cfg, eval_crop_pct=crop)
        ax.imshow(view_tensor(tf(image))); ax.set_title(f'Eval crop {crop}')
    for ax in axs: ax.axis('off')
fig.tight_layout(); fig.savefig(out/'crop_comparison.png', dpi=140); plt.show()


## 5. Duplicate audit
Exact matches below mean identical file bytes. Re-encoded copies need perceptual review. dHash candidates use Hamming distance with a band index (avoids an all-pairs scan). With threshold 4, five disjoint bands guarantee that any hash pair within threshold shares a band. Flat/simple images can create many false positives; candidate comparisons and output are capped. Cross-label matches may be label problems or legitimate related classes. Cross-split matches can bias evaluation; inspect and document them before deciding how to handle them.


In [ ]:
if HASH_EXACT:
    exact = good[good.duplicated('sha256', keep=False)].sort_values('sha256')
    exact.to_csv(out/'exact_duplicate_images.csv', index=False)
    print('Images in exact duplicate groups:', len(exact))
    if len(exact):
        display(exact[['split','path','label','sha256']].head(30))
        gallery(exact.head(20).to_dict('records'), 'exact_duplicates.png')

if RUN_NEAR_DUPLICATES:
    assert 0 <= NEAR_DISTANCE <= 8, 'Choose a distance from 0 through 8.'
    from collections import defaultdict
    bands = NEAR_DISTANCE + 1
    boundaries = np.linspace(0,64,bands+1,dtype=int)
    index = defaultdict(list)
    items = good.reset_index(drop=True)
    hashes = [int(x,16) for x in items.dhash]
    pairs = []
    comparisons = 0
    comparison_cap = 2_000_000
    capped = False
    for i, value in enumerate(tqdm(hashes, desc='Near duplicate candidates')):
        keys = [(b, (value >> int(boundaries[b])) & ((1 << int(boundaries[b+1]-boundaries[b]))-1)) for b in range(bands)]
        candidates = set()
        for key in keys: candidates.update(index[key])
        for j in sorted(candidates):
            comparisons += 1
            if comparisons > comparison_cap:
                capped = True; break
            distance = (value ^ hashes[j]).bit_count()
            if distance <= NEAR_DISTANCE:
                a,b = items.iloc[j], items.iloc[i]
                pairs.append(dict(path_a=a.path, split_a=a.split, label_a=int(a.label), path_b=b.path, split_b=b.split, label_b=int(b.label), distance=distance))
                if len(pairs) >= MAX_NEAR_PAIRS:
                    capped = True; break
        if capped: break
        for key in keys: index[key].append(i)
    near = pd.DataFrame(pairs, columns=['path_a','split_a','label_a','path_b','split_b','label_b','distance'])
    near.to_csv(out/'near_duplicate_pairs.csv', index=False)
    print('Candidate pairs:', len(near), 'Capped:', capped, '(sample-only unless complete scan selected)')
    display(near.head(20))
    pair_rows = []
    for pair in near.head(10).itertuples(index=False):
        pair_rows.extend([dict(path=pair.path_a, split=pair.split_a, label=pair.label_a), dict(path=pair.path_b, split=pair.split_b, label=pair.label_b)])
    gallery(pair_rows, 'near_duplicate_pairs.png', columns=2)


## 6. Record decisions before another training run
Use the generated galleries and CSVs to answer:

- Does cropping preserve the object, especially for the worst-performing kinds of images?
- Do validation images differ in resolution, aspect ratio, or framing?
- Are duplicate groups or suspicious labels concentrated in particular classes?
- Is there a reason to compare a less aggressive training crop, another evaluation crop, or class-specific error patterns?

Change one factor at a time on a fixed internal holdout and equal training budget. Keep holdout images and their duplicates out of prototype fitting. A random five-per-class split alone does not prevent near-duplicate leakage. Do not remove images automatically using the proxy scores here.

This notebook does not measure model accuracy. For an error gallery, use a checkpoint trained without the internal holdout (including head/prototype fitting), and save per-image true label, predicted label, confidence and top-five predictions. Existing full-data checkpoints have already seen those training images, so their predictions cannot provide an unbiased internal holdout evaluation. Official validation error labels remain unavailable locally.

Outputs are saved under `OUTPUT_DIR`; share the CSVs and selected galleries for the next analysis. Rerunning overwrites files with the same names, so use a new output folder for a different audit.
